# Aula

In [1]:
import os
import kagglehub
import pandas as pd

# kagglehub.login()

# Baixa e extrai os arquivos no cache oculto do sistema (sem criar arquivos na pasta do projeto)
caminho         = kagglehub.dataset_download('dansbecker/melbourne-housing-snapshot')
melb_data_path  = (os.path.join(caminho, 'melb_data.csv'))

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the data
data = pd.read_csv(melb_data_path)

# Select target
y = data.Price

# To keep things simple, we'll use only numerical predictors
melb_predictors = data.drop(['Price'], axis=1)
X = melb_predictors.select_dtypes(exclude=['object'])

# Divide data into training and validation subsets
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

## Define Function to Measure Quality of Each Approach

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=10, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

## Approach 1 - Drop Columns with Missing Values

In [4]:
# Get name of columns with missing values
cols_with_missing = [col for col in X_train.columns if X_train[col].isnull().any()]

#Drop columns in training and validation data
reduced_X_train = X_train.drop(cols_with_missing, axis=1)
reduced_X_valid = X_valid.drop(cols_with_missing, axis=1)

print('MAE from Approach 1:', score_dataset(reduced_X_train, reduced_X_valid, y_train, y_valid))

MAE from Approach 1: 183550.22137772635


## Approach 2 - Imputation

In [6]:
# SimpleImputer to replace missing values with the mean value along each column.
from sklearn.impute import SimpleImputer

# Imputation
my_imputer      = SimpleImputer()
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train)) 
imputed_X_valid = pd.DataFrame(my_imputer.transform(X_valid))

# Imputation removed column names; put them back
imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns


print('MAE from Approach 2:', score_dataset(imputed_X_train, imputed_X_valid, y_train, y_valid))

MAE from Approach 2: 178166.46269899711


## Approach 3 - An Extension to imputation

In [13]:
# Copy to avoid changing original data

X_train_plus = X_train.copy()
X_valid_plus = X_valid.copy()

# Make new columns indicationg what will be imputed
for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_valid_plus[col + '_was_missing'] = X_valid_plus[col].isnull()
    
# Imputation
my_imputer              = SimpleImputer()
imputed_X_train_plus    = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
imputed_X_valid_plus    = pd.DataFrame(my_imputer.transform(X_valid_plus))

# Imputations removed column names; put them back
imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_valid_plus.columns = X_valid_plus.columns

print('Mae from Approach 3:', score_dataset(imputed_X_train_plus, imputed_X_valid_plus, y_train, y_valid))

Mae from Approach 3: 178927.503183954


# Exercícios

In [20]:
train_csv   = os.path.join(kagglehub.competition_download('home-data-for-ml-course'), 'train.csv')
test_csv    = os.path.join(kagglehub.competition_download('home-data-for-ml-course'), 'test.csv')

X_full      = pd.read_csv(train_csv, index_col='Id')
X_test_full = pd.read_csv(test_csv, index_col='Id')

# Remove rows with missing target, separate target from predictors
X_full.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X_full.SalePrice
X_full.drop(['SalePrice'], axis=1, inplace=True)

# Shape of training data (num_rows, num_columns)
print(X_train.shape)

# Number of missing values in each column of training data
missing_val_count_by_column = (X_train.isnull().sum())
print(missing_val_count_by_column[missing_val_count_by_column > 0])

(10864, 12)
Car               49
BuildingArea    5156
YearBuilt       4307
dtype: int64


In [21]:
# Fill in the line below: How many rows are in the training data?
num_rows = 1168

# Fill in the line below: How many columns in the training data have missing values?
num_cols_with_missing = 3

# Fill in the line below: How many missing entries are contained in all of the training data?
tot_missing = 276

In [22]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

In [25]:
cols_with_missing_train = [col for col in X_full.columns if X_full[col].isnull().any()]
cols_with_missing_valid = [col for col in X_test_full.columns if X_test_full[col].isnull().any()]

reduced_X_train = X_full.drop(cols_with_missing_train, axis=1)
reduced_X_valid = X_test_full.drop(cols_with_missing_valid, axis=1)